In [1]:
MARGIN = 10  # pixels
FONT_SIZE = 1
FONT_THICKNESS = 1
HANDEDNESS_TEXT_COLOR = (88, 205, 54)  # vibrant green
MAX_HANDS = 2
CAMERA = 0
FPS = 30
REC_TEST = False
frame_idx = 0

DATA_HEADER = [
    "th1",
    "th2",
    "th3",
    "in1",
    "in2",
    "in3",
    "mi1",
    "mi2",
    "mi3",
    "ri1",
    "ri2",
    "ri3",
    "pi1",
    "pi2",
    "pi3",
    "hand_side",
]


LABEL_HEADER = [
    # finger labels
    # 1 = open, 0 = folded
    "th_open",
    "in_open",
    "mi_open",
    "ri_open",
    "pi_open",
    "ti_touch",
    "tm_touch",
]

In [2]:
import cv2
import mediapipe as mp
import numpy as np

from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# Utils

In [3]:
def create_hand_detector(model_path: str = "hand_landmarker.task", num_hands: int = 1):
    """
    MediaPipe HandLandmarker 생성.
    """
    base_options = python.BaseOptions(model_asset_path=model_path)
    options = vision.HandLandmarkerOptions(
        base_options=base_options,
        num_hands=num_hands,
        running_mode=mp.tasks.vision.RunningMode.VIDEO,
    )
    detector = vision.HandLandmarker.create_from_options(options)
    return detector

def get_drawing_utils():
    """
    랜드마크 시각화를 위한 연결 정보 반환.
    """
    return vision.HandLandmarksConnections.HAND_CONNECTIONS

def capture_frame(cap: cv2.VideoCapture):
    """
    웹캠에서 프레임 1장을 읽어 반환.
    실패 시 (False, None) 반환.
    """
    ret, frame = cap.read()
    frame = cv2.flip(frame, 1)
    if not ret:
        return False, None
    return True, frame

def convert_bgr_to_mp_image(bgr_frame: np.ndarray):
    """
    OpenCV BGR 이미지를 RGB 및 MediaPipe Image로 변환.
    """
    rgb_frame = cv2.cvtColor(bgr_frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
    return rgb_frame, mp_image

def detect_hand_landmarks(detector, bgr_frame: np.ndarray):
    """
    BGR 프레임을 받아 MediaPipe HandLandmarker로 손 랜드마크 검출.
    반환:
      - rgb_frame
      - detection_result
    """
    global frame_idx
    rgb_frame, mp_image = convert_bgr_to_mp_image(bgr_frame)
    timestamp_ms = int(frame_idx * 1000 / FPS)
    # detection_result = detector.detect(mp_image)
    detection_result = detector.detect_for_video(mp_image, timestamp_ms)
    frame_idx += 1


    hand_landmarks_list = getattr(detection_result, "hand_landmarks", [])
    handedness_list = getattr(detection_result, "handedness", [])

    for idx in range(len(hand_landmarks_list)):
        hand_label = handedness_list[idx][0].category_name

        if hand_label == "Right":
            handedness_list[idx][0].category_name = "Left"
        elif hand_label == "Left":
            handedness_list[idx][0].category_name = "Right"

    return rgb_frame, detection_result

def draw_landmarks_on_image(rgb_image, detection_result, hand_connections):
    hand_landmarks_list = detection_result.hand_landmarks
    handedness_list = detection_result.handedness
    annotated_image = np.copy(rgb_image)
    image_h, image_w, _ = annotated_image.shape

    for idx, hand_landmarks in enumerate(hand_landmarks_list):
        # 관절 연결선 그리기
        for connection in hand_connections:
            start_idx, end_idx = connection.start, connection.end
            start_lm = hand_landmarks[start_idx]
            end_lm = hand_landmarks[end_idx]
            start_point = (int(start_lm.x * image_w), int(start_lm.y * image_h))
            end_point = (int(end_lm.x * image_w), int(end_lm.y * image_h))
            cv2.line(annotated_image, start_point, end_point, (0, 255, 255), 2)

        # 관절 포인트 그리기
        for lm in hand_landmarks:
            px = int(lm.x * image_w)
            py = int(lm.y * image_h)
            cv2.circle(annotated_image, (px, py), 4, (255, 80, 80), -1)

        if idx < len(handedness_list):
            handedness = handedness_list[idx]
            x_coordinates = [landmark.x for landmark in hand_landmarks]
            y_coordinates = [landmark.y for landmark in hand_landmarks]
            text_x = int(min(x_coordinates) * image_w)
            text_y = int(min(y_coordinates) * image_h) - MARGIN

            cv2.putText(
                annotated_image,
                f"{handedness[0].category_name}",
                (text_x, text_y),
                cv2.FONT_HERSHEY_DUPLEX,
                FONT_SIZE,
                HANDEDNESS_TEXT_COLOR,
                FONT_THICKNESS,
                cv2.LINE_AA,
            )
    return annotated_image

# MODEL

In [4]:
import tensorflow as tf
import pickle


def load_model(model_file="gesture_model.keras", scalar_file="gesture_scaler.pkl"):
    model = tf.keras.models.load_model(model_file, safe_mode=False, compile=True)
    with open(scalar_file, "rb") as f:
        scaler = pickle.load(f)

    return model, scaler



In [5]:
def predict_gesture(model, scaler, feature):

    x = np.array(feature, dtype=np.float32).reshape(1, -1)

    x = scaler.transform(x)

    pred = model(x, traning = False).numpy()[0]

    return pred


def predict_hand_gesture_by_dnn(detection_result, model, scalar):

    left_result = None
    right_result = None
    center = None

    hand_landmarks_list = getattr(detection_result, "hand_landmarks", [])
    handedness_list = getattr(detection_result, "handedness", [])

    if not hand_landmarks_list:
        return {"left": None, "right": None}
    

    for idx in range(len(hand_landmarks_list)):
        hand_landmarks = hand_landmarks_list[idx]
        hand_label = handedness_list[idx][0].category_name

        hand_side = -1
        if hand_label == "Right":
            hand_side = 1
        elif hand_label == "Left":
            hand_side = 0
        # hand_side = hand_side * 100

        joint = np.array([[lm.x, lm.y, lm.z] for lm in hand_landmarks])

        v1 = joint[[0,1,2,3,0,5,6,7,0,9,10,11,0,13,14,15,0,17,18,19],:]
        v2 = joint[[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20],:]

        v = v2 - v1
        v = v / np.linalg.norm(v, axis=1)[:, np.newaxis]

        angle = np.arccos(np.einsum(
            'nt,nt->n',
            v[[0,1,2,4,5,6,8,9,10,12,13,14,16,17,18],:],
            v[[1,2,3,5,6,7,9,10,11,13,14,15,17,18,19],:]
        ))

        angle = np.degrees(angle)

        full_data = angle

        palm_size = np.linalg.norm(joint[17] - joint[5]) + 1e-6

        full_data = np.append(full_data, hand_side)

        data = np.array([full_data], dtype=np.float32)
    
        pred = predict_gesture(model, scalar, data)
        # label = int(results[0][0])
        joint = np.array([[lm.x, lm.y, lm.z] for lm in hand_landmarks])
        v = joint[[0, 5, 9, 13, 17]] - 0.5

        if hand_label == "Right":
            center = v.mean(axis=0)
            right_result = pred
        elif hand_label == "Left":
            left_result = pred

    return {
        "left": left_result,
        "right": right_result,
        "center": center
    }


# Something.. be used for multi sequence key input...

# Result Test

In [6]:
model, scalar = load_model()
from Gesture import GestureController


cap = cv2.VideoCapture(CAMERA)
if not cap.isOpened():
    raise RuntimeError(f"웹캠을 열 수 없습니다. camera_index={CAMERA}")

detector = create_hand_detector(model_path="hand_landmarker.task", num_hands=MAX_HANDS)
hand_connections = get_drawing_utils()
gesture = GestureController()
try:
    while True:
        ret, frame = capture_frame(cap)
        if not ret:
            print("프레임을 읽지 못했습니다. 종료합니다.")
            break

        rgb_frame, detection_result = detect_hand_landmarks(detector, frame)
        result = predict_hand_gesture_by_dnn(detection_result, model, scalar)
        annotated_image = draw_landmarks_on_image(rgb_frame, detection_result, hand_connections)

        # OpenCV는 BGR 기준으로 표시하므로 RGB -> BGR로 변환
        display_image = cv2.cvtColor(annotated_image, cv2.COLOR_RGB2BGR)

        font = cv2.FONT_HERSHEY_SIMPLEX
        line_height = 26

        # gesture_hid.emit(result)
        gesture.update(result, result.get('center', None))

        # ===== LEFT HAND =====
        lines_left = ["LEFT HAND"]

        for name, p in zip(LABEL_HEADER, [0] * 7 if result['left'] is None else result['left']):
            lines_left.append(f"{name:<10} : {p:>6.2%}")

        for i, line in enumerate(lines_left):
            y = 60 + i * line_height
            cv2.putText(
                display_image,
                line,
                (10, y),
                font,
                0.6,
                (0,255,0),
                2,
                cv2.LINE_AA
            )


        # ===== RIGHT HAND =====
        lines_right = ["RIGHT HAND"]

        for name, p in zip(LABEL_HEADER, [0] * 7 if result['right'] is None else result['right']):
            lines_right.append(f"{name:<10} : {p:>6.2%}")

        for i, line in enumerate(lines_right):
            y = 60 + i * line_height
            cv2.putText(
                display_image,
                line,
                (350, y),
                font,
                0.6,
                (255,255,0),
                2,
                cv2.LINE_AA
            )


        cv2.imshow("MediaPipe Hand Landmarks", display_image)

        # ESC 또는 q 입력 시 종료
        key = cv2.waitKey(1) & 0xFF
        if key == 27 or key == ord("q"):
            break
        
finally:
    cap.release()
    cv2.waitKey(-1)
    cv2.destroyAllWindows()


0 0
4 26
13 21
8 16
7 13
20 31
4 8
2 4
2 2
8 0
0 0
0 -1
0 0
0 0
0 0
-3 -1
-1 0
-1 0
-2 0
-2 0
-9 -3
-1 0
-1 0
-1 0
-1 0
0 0
0 0
0 0
1 0
12 0
3 0
4 0
5 0
5 -1
5 -2
5 -4
5 -3
5 -3
17 -10
5 0
5 2
4 2
4 3
3 4
2 4
1 4
-5 18
-5 4
-8 3
-7 2
-9 0
-9 0
-9 -1
-8 -3
-31 -19
-11 -8
-6 -7
-3 -9
-2 -7
0 -7
0 -5
-1 -2
-3 0
-5 5
-19 24
-1 8
0 6
2 4
4 2
5 1
5 0
20 -9
5 -5
4 -7
4 -7
3 -9
8 -30
1 -7
3 -2
3 0
3 2
3 4
3 5
2 25
0 6
-3 4
-5 1
0 0
-8 -1
-49 -38
-4 2
0 6
0 8
1 9
4 22
2 11
1 3
0 2
1 2
0 1
0 2
0 2
0 0
0 3
0 3
1 3
2 2
9 4
8 -5
0 0
-1 -2
-1 -2
-2 -4
5 0
4 0
7 1
0 0
7 -1
9 1
7 2
2 24
-7 4
-18 -2
-10 -6
0 0
-9 -7
-12 -37
4 -16
6 -7
8 -3
7 1
6 4
5 7
-7 41
-6 20
-8 10
-9 7
-10 3
-7 0
29 -33
-5 1
-4 2
-1 2
0 1
3 2
6 0
35 -25
4 -9
3 -12
0 -11
-1 -10
-3 -7
-5 -4
-6 -1
-7 2
-17 19
-3 11
1 8
4 6
5 5
5 3
7 -1
0 0
7 -3
9 -29
3 -21
-4 -11
-6 -8
-6 -5
-19 0
-8 13
0 9
1 9
2 8
3 8
3 6
4 4
10 -10
0 -12
-6 -31
